# 10 · Integration metrics (scib)

*sc-rnaseq-wf, re-instantiated step by step for `team-jakobsson` (`jakobsson-pmdbs-sn-rnaseq`)*

| | |
| :- | :- |
| WDL task | [`artifact_metrics`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/workflows/cohort_analysis/cohort_analysis.wdl) |
| Script | [`artifact_metrics`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/docker/sc_tools/scripts/main/artifact_metrics) → `sc-artifact-metrics` (already run inside `sc-annotate`) |

[scib-metrics](https://scib-metrics.readthedocs.io/) scores how well each embedding removes batch effects while keeping biological structure. `sc-annotate` already computed the report with the same function (`get_artifact_metrics`); this notebook reads it. Section 3 re-runs it with `sc-artifact-metrics`.

Embeddings compared: `Unintegrated` (`X_pca`), `X_scVI`, `X_pca_harmony`. Labels: `C_scANVI`. Batch: `batch_id`.

| Group | Metrics | High means |
| :- | :- | :- |
| Bio conservation | NMI / ARI of k-means clusters vs. labels, silhouette label, cLISI | cell types stay separate |
| Batch correction | silhouette batch / BRAS, iLISI, kBET, graph connectivity, PCR comparison | batches are mixed within cell types |

Isolated-label metrics are turned off. Scores are not min–max scaled, so they are comparable across runs. If no cell type contains more than one batch, batch-correction metrics are undefined and only bio conservation is reported.

> `sc-artifact-metrics` subsamples to 500,000 cells before computing these (`sc-annotate` does not); a single team's dataset is below that.

**Inputs**
- `${cohort_id}.final.h5ad`

**Outputs** (in `cfg.RUN_DIR`, named as in the WDL)
- `scib_report_dir/${cohort_id}.scib_report.csv`
- `scib_report_dir/${cohort_id}.scib_results.svg`

**Previous:** [09_annotate](09_annotate.ipynb) · **Next:** [11_plot_groups_and_feats](11_plot_groups_and_feats.ipynb)

In [ ]:
import wf_config as cfg
import pandas as pd
import scanpy as sc
from IPython.display import Image, display

cfg.RUN_DIR.mkdir(parents=True, exist_ok=True)
print("run directory:", cfg.RUN_DIR)

## 1. The report

In [ ]:
from IPython.display import SVG

report_dir = cfg.RUN_DIR / "scib_report_dir"
report = pd.read_csv(report_dir / f"{cfg.COHORT_ID}.scib_report.csv", index_col=0)
display(report)
display(SVG(filename=str(report_dir / f"{cfg.COHORT_ID}.scib_results.svg")))

## 2. Compare with the curated release

In [ ]:
released = cfg.curated("scib_report.csv")
if released.exists():
    display(pd.concat({"this run": report, "release": pd.read_csv(released, index_col=0)}, axis=0))
else:
    print(f"{released.name} not in the curated release")

## 3. Optional: re-run `sc-artifact-metrics`

For example, after changing the labels or the embeddings in `final.h5ad`.

In [ ]:
RERUN = False

if RERUN:
    cfg.run(
        "sc-artifact-metrics",
        "--predictions-key", cfg.SCANVI_PREDICTIONS_KEY,
        "--batch-key", cfg.BATCH_KEY,
        "--adata-input", cfg.out("final.h5ad"),
        "--output-report-dir", "scib_report_rerun",
    )
    display(pd.read_csv(cfg.RUN_DIR / "scib_report_rerun" / "scib_report.csv", index_col=0))

**Previous:** [09_annotate](09_annotate.ipynb) · **Next:** [11_plot_groups_and_feats](11_plot_groups_and_feats.ipynb)